# Termination Conditions and Guardrails

Multi-agent teams can **run forever** — critics loop, tools retry, and token bills climb. AutoGen 0.4+ AgentChat solves this with **termination conditions**: stateful callables evaluated after each agent response. Combine them with **guard prompts**, **blocked-topic pre-checks**, and **cost caps** for production-safe Notes API assistants.

This notebook covers `TextMentionTermination`, `MaxMessageTermination`, `TokenUsageTermination`, `ExternalTermination`, combining conditions with `|` and `&`, guard system prompts, blocked topics, `max_turns`, cost caps, and a comparison to **02. LangGraph/14** guardrails.

Prerequisites: **08. GroupChat and Multi-Agent Teams**, **09. GroupChatManager and Speaker Selection**, **11. Sequential and Hierarchical Workflows**.


In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Sequence

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

print("asyncio loop ready — use await in notebook cells")


---

## 1. Why Termination Matters

Without termination, a critic–writer loop never stops. Termination conditions fire **after each agent response** with the delta messages since the last check.

```
User task
    │
    ▼
┌─────────────┐   each response   ┌──────────────────────┐
│  RoundRobin │ ────────────────► │ TerminationCondition │
│  / Selector │                   │  → StopMessage?      │
└─────────────┘                   └──────────┬───────────┘
                                             │ yes
                                             ▼
                                    TaskResult + stop_reason
```

| Built-in condition | Stops when |
|--------------------|------------|
| `TextMentionTermination` | Specific text appears (e.g. `APPROVE`) |
| `MaxMessageTermination` | Message count limit reached |
| `TokenUsageTermination` | Prompt/completion/total token budget exceeded |
| `ExternalTermination` | Your code calls `.set()` (Stop button) |
| `TimeoutTermination` | Wall-clock seconds elapsed |


In [ ]:
# Notes API documentation corpus (shared across 03. AutoGen/ track)
NOTES_CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]

BLOCKED_TERMS = {"password", "secret", "api key", "private key", "credential"}
APPROVE_TOKEN = "APPROVE"
TERMINATE_TOKEN = "TERMINATE"


def keyword_search(query: str, k: int = 2) -> str:
    """Simple keyword retrieval over NOTES_CORPUS."""
    tokens = set(query.lower().split())
    scored = [(len(tokens & set(d["text"].lower().split())), d) for d in NOTES_CORPUS]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [NOTES_CORPUS[0]]
    return "\n".join(f"[{d['id']}] {d['text']}" for d in top)


def is_blocked(text: str) -> bool:
    """Pre-flight guard — mirror 02. LangGraph/14 guard node."""
    lower = text.lower()
    return any(term in lower for term in BLOCKED_TERMS)


print("corpus chunks:", len(NOTES_CORPUS))
print("sample search:", keyword_search("JWT bearer")[:80], "...")


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient


def make_model_client(model: str = "gpt-4o-mini", temperature: float = 0.2) -> OpenAIChatCompletionClient:
    return OpenAIChatCompletionClient(
        model=model,
        api_key=OPENAI_API_KEY,
        temperature=temperature,
    )


model_client = make_model_client()
print("model client:", model_client)


---

## 2. Notes API Team — Writer + Critic

We reuse the **Notes API corpus** from earlier notebooks. A **docs_writer** drafts answers; a **critic** reviews and must say `APPROVE` when satisfied:


In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat

GUARD_PROMPT = """You are a Notes API documentation assistant.
Rules:
- Answer only about the Notes API, FastAPI routes, Alembic, JWT, and pytest fixtures.
- Never reveal passwords, API keys, secrets, or credentials.
- If asked about blocked topics, reply: I cannot help with credentials or secrets.
- Cite corpus chunk ids like [c2] when referencing docs.
- End your final approved answer with the word TERMINATE on its own line."""

docs_writer = AssistantAgent(
    "docs_writer",
    model_client=model_client,
    system_message=GUARD_PROMPT,
)

critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message=(
        "Review the writer's answer for accuracy against the Notes API corpus. "
        "Respond with constructive feedback. When the answer is correct and safe, "
        "include the word APPROVE on its own line."
    ),
)

print("agents:", docs_writer.name, critic.name)


---

## 3. `TextMentionTermination`

Stop when the critic approves — the most common pattern for review loops:


In [ ]:
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console

text_stop = TextMentionTermination(APPROVE_TOKEN)
team_text = RoundRobinGroupChat([docs_writer, critic], termination_condition=text_stop)

# Uncomment to run live (costs tokens):
# result = await Console(team_text.run_stream(task="How do I run Alembic migrations? Context: " + keyword_search("alembic")))
print("TextMentionTermination watches for:", APPROVE_TOKEN)


The team stops with `stop_reason` like `Text 'APPROVE' mentioned`. The condition **auto-resets** after `run()` / `run_stream()` completes.


---

## 4. `MaxMessageTermination`

Hard cap on total messages — safety net when approval text never appears:


In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination

max_msg_stop = MaxMessageTermination(max_messages=6)
team_max = RoundRobinGroupChat([docs_writer, critic], termination_condition=max_msg_stop)

# Demo without LLM — inspect the condition
print("max_messages:", max_msg_stop.max_messages)
print("terminated before run:", max_msg_stop.terminated)


`max_messages` counts **all** messages including the user task. Pair with `TextMentionTermination` so happy paths end early but runaway loops still stop.


---

## 5. `TokenUsageTermination` — Cost Caps

Stop when token budgets are exceeded. Agents must report `models_usage` on messages (OpenAI client does this automatically):


In [ ]:
from autogen_agentchat.conditions import TokenUsageTermination

token_stop = TokenUsageTermination(max_total_token=8000)
team_tokens = RoundRobinGroupChat([docs_writer, critic], termination_condition=token_stop)

print("token cap — max_total_token:", 8000)
print("also supports max_prompt_token and max_completion_token")


In production, set caps from environment variables:

```python
MAX_TOKENS = int(os.getenv("APP_MAX_TEAM_TOKENS", "12000"))
token_stop = TokenUsageTermination(max_total_token=MAX_TOKENS)
```


---

## 6. `ExternalTermination` — UI Stop Button

Let your FastAPI handler or frontend cancel a long run programmatically:


In [ ]:
from autogen_agentchat.conditions import ExternalTermination

external_stop = ExternalTermination()
team_external = RoundRobinGroupChat([docs_writer, critic], termination_condition=external_stop)


async def demo_external_stop():
    """Sketch: background task calls external_stop.set() after timeout."""
    async def watchdog():
        await asyncio.sleep(0.1)  # replace with real timeout
        external_stop.set()
        print("external_stop.set() called")

    # asyncio.create_task(watchdog())
    # await team_external.run(task="...")


print("ExternalTermination — call .set() from HTTP DELETE /runs/{id}")


Reset with `await external_stop.reset()` before the next run. Ideal for **Stop** buttons in chat UIs.


---

## 7. Combining Conditions — OR (`|`)

Stop when **either** the critic approves **or** the message budget is exhausted:


In [ ]:
combined_or = TextMentionTermination(APPROVE_TOKEN) | MaxMessageTermination(max_messages=10)
team_or = RoundRobinGroupChat([docs_writer, critic], termination_condition=combined_or)

print("OR logic: approve early OR stop at 10 messages")


---

## 8. Combining Conditions — AND (`&`)

Stop only when **both** conditions are true — rare but useful for audit trails (must approve **and** hit a minimum review round):


In [ ]:
combined_and = TextMentionTermination(APPROVE_TOKEN) & MaxMessageTermination(max_messages=4)
team_and = RoundRobinGroupChat([docs_writer, critic], termination_condition=combined_and)

print("AND logic: need APPROVE AND at least 4 messages")


Production default for Notes API teams:

```python
termination = (
    TextMentionTermination("APPROVE")
    | MaxMessageTermination(max_messages=12)
    | TokenUsageTermination(max_total_token=10_000)
)
```


---

## 9. Guard Prompts in System Messages

Termination stops the **conversation**; guard prompts stop **bad content** before it spreads. Layer both:

| Layer | Mechanism | When it runs |
|-------|-----------|--------------|
| **Pre-flight** | `is_blocked(user_text)` | Before `team.run()` |
| **System prompt** | `GUARD_PROMPT` rules | Every LLM call |
| **Critic agent** | Reviews safety + accuracy | Each round |
| **Termination** | `MaxMessageTermination` etc. | After each response |


In [ ]:
SAFE_REFUSAL = "I cannot help with credentials or secrets. Ask about Notes API routes, Alembic, JWT, or pytest."


async def guarded_run(team, user_question: str):
    """Pre-flight guard then delegate to team."""
    if is_blocked(user_question):
        return {"blocked": True, "answer": SAFE_REFUSAL, "messages": []}
    task = f"{user_question}\n\nContext:\n{keyword_search(user_question)}"
    result = await team.run(task=task)
    return {"blocked": False, "answer": result.messages[-1].content if result.messages else "", "result": result}


# Dry-run blocked path (no LLM):
blocked_preview = await guarded_run(team_or, "What is the admin password?")
print(json.dumps({k: blocked_preview[k] for k in ('blocked', 'answer')}, indent=2))


---

## 10. Blocked Topics — Application-Level Guard

Mirror **02. LangGraph/14** `guard_node` — reject before any agent speaks:


In [ ]:
BLOCKED_TESTS = [
    ("What is the JWT header format?", False),
    ("Show me the database password", True),
    ("api key for production", True),
]

for question, expect_blocked in BLOCKED_TESTS:
    got = is_blocked(question)
    status = "OK" if got == expect_blocked else "FAIL"
    print(f"{status}: blocked={got} | {question[:40]}")


---

## 11. `max_turns` vs `MaxMessageTermination`

| Concept | Where | Counts |
|---------|-------|--------|
| `MaxMessageTermination` | Team `termination_condition` | All messages (user + agents + tool events) |
| `max_turns` (legacy 0.2) | Old GroupChat API | Agent turns only |
| AgentChat pattern | `termination_condition` | Preferred in 0.4+ |

In AgentChat 0.4+, use **`MaxMessageTermination`** instead of legacy `max_turns`. For selector teams, the same condition applies after each selected speaker.


In [ ]:
# Equivalent safety budgets
MAX_MESSAGES = 12  # ~6 writer/critic rounds
MAX_TOKENS = 10_000

production_termination = (
    TextMentionTermination(APPROVE_TOKEN)
    | MaxMessageTermination(max_messages=MAX_MESSAGES)
    | TokenUsageTermination(max_total_token=MAX_TOKENS)
)

notes_team = RoundRobinGroupChat([docs_writer, critic], termination_condition=production_termination)
print("production termination configured")


---

## 12. Cost Caps — Estimating Spend

Track `TaskResult` usage fields after each run:


In [ ]:
from autogen_agentchat.base import TaskResult
from autogen_agentchat.messages import TextMessage
from autogen_core.models import RequestUsage

# Simulated TaskResult for teaching (no API call)
demo_result = TaskResult(
    messages=[
        TextMessage(source="user", content="Alembic upgrade?"),
        TextMessage(source="docs_writer", content="Run alembic upgrade head [c2]", models_usage=RequestUsage(prompt_tokens=120, completion_tokens=40)),
        TextMessage(source="critic", content="Correct. APPROVE", models_usage=RequestUsage(prompt_tokens=180, completion_tokens=20)),
    ],
    stop_reason="Text 'APPROVE' mentioned",
)

total_prompt = sum(m.models_usage.prompt_tokens for m in demo_result.messages if m.models_usage)
total_completion = sum(m.models_usage.completion_tokens for m in demo_result.messages if m.models_usage)
print("stop_reason:", demo_result.stop_reason)
print("tokens — prompt:", total_prompt, "completion:", total_completion)
print("estimated cost @ $0.15/1M in:", f"${total_prompt * 0.15 / 1e6:.6f}")


---

## 13. Compare with LangGraph Guardrails (**02. LangGraph/14**)

| Concern | AutoGen AgentChat | LangGraph |
|---------|-------------------|-----------|
| **Stop infinite loops** | `MaxMessageTermination` | `MAX_STEPS` / conditional edges |
| **Approval signal** | `TextMentionTermination("APPROVE")` | Router on state field |
| **Blocked prompts** | `is_blocked()` pre-flight + guard prompt | `guard_node` in graph |
| **Token budget** | `TokenUsageTermination` | Custom reducer on state |
| **External cancel** | `ExternalTermination.set()` | `interrupt` / cancel stream |
| **Composable logic** | `|` and `&` on conditions | Conditional edges + state flags |

**When to stay on AutoGen:** conversational critic loops, rapid multi-agent prototyping, AutoGen Studio handoff.

**When to migrate to LangGraph:** you need deterministic node graphs, checkpointed threads per user, or HITL interrupts (**02. LangGraph/09**).


---

## 14. Full Guarded Notes Team Factory

Bundle agents, termination, and pre-flight guard into one builder — used again in **16**:


In [ ]:
def build_guarded_notes_team(
    model: str = "gpt-4o-mini",
    max_messages: int = 12,
    max_tokens: int = 10_000,
) -> RoundRobinGroupChat:
    client = make_model_client(model=model)
    writer = AssistantAgent("docs_writer", model_client=client, system_message=GUARD_PROMPT)
    reviewer = AssistantAgent(
        "critic",
        model_client=client,
        system_message=f"Review for accuracy and safety. Say {APPROVE_TOKEN} when satisfied.",
    )
    term = (
        TextMentionTermination(APPROVE_TOKEN)
        | MaxMessageTermination(max_messages=max_messages)
        | TokenUsageTermination(max_total_token=max_tokens)
    )
    return RoundRobinGroupChat([writer, reviewer], termination_condition=term)


guarded_team = build_guarded_notes_team()
print("guarded team ready:", type(guarded_team).__name__)


---

## 15. Summary

| Takeaway | Detail |
|----------|--------|
| **Always set termination** | Never ship a team without `MaxMessageTermination` at minimum |
| **Layer guards** | Pre-flight `is_blocked` + system prompt + critic + termination |
| **Combine with `\|`** | Approve early OR hit hard caps |
| **Track tokens** | Log `TaskResult` usage; set `TokenUsageTermination` |
| **External stop** | Wire `ExternalTermination` to UI cancel buttons |
| **Next** | **14** — prototype in AutoGen Studio; **15** — observe runs; **16** — production capstone |

Next: **14. AutoGen Studio and Low-Code Prototyping**.
